In [ ]:
from pathlib import Path
import pandas as pd

# Locate the project folder
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Load cleaned data
data_path = project_root / "data" / "hotel_bookings_clean.csv"
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Total columns:", len(df.columns))
print("First 10 columns:")
print(df.columns[:10].tolist())

In [ ]:
# Create a basic data dictionary
data_dictionary = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})

# Assign a role to each feature
data_dictionary["Role"] = "Predictor"

data_dictionary.loc[
    data_dictionary["Feature"] == "is_canceled",
    "Role"
] = "Target"

data_dictionary.loc[
    data_dictionary["Feature"] == "row_id",
    "Role"
] = "Identifier"

# Add columns for later documentation
data_dictionary["Description"] = ""
data_dictionary["Cleaning Notes"] = ""

display(data_dictionary.head(10))

print("Dictionary shape:", data_dictionary.shape)

In [ ]:
print(data_dictionary["Role"].value_counts())

print("\nTarget and identifier:")
display(
    data_dictionary[
        data_dictionary["Role"].isin(
            ["Target", "Identifier"]
        )
    ]
)

In [ ]:
# Check all columns in the current cleaned dataset
print("Current dataset columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

# Check the new season feature
print("\nArrival season exists:",
      "arrival_season" in df.columns)

# Check possible leakage columns
leakage_columns = [
    "reservation_status",
    "reservation_status_date",
    "booking_changes",
    "assigned_room_type",
    "days_in_waiting_list"
]

print("\nPotential leakage columns still present:")
print([col for col in leakage_columns if col in df.columns])

# Check row ID
print("\nUnique row IDs:", df["row_id"].is_unique)
print("Missing values:", df.isna().sum().sum())

In [ ]:
# Define target and identifier
target_col = "is_canceled"
id_col = "row_id"

# Define features excluded from prediction
excluded_features = [
    "reservation_status",
    "reservation_status_date",
    "booking_changes",
    "assigned_room_type",
    "days_in_waiting_list"
]

# Reset roles
data_dictionary["Role"] = "Review"

# Assign target
data_dictionary.loc[
    data_dictionary["Feature"] == target_col,
    "Role"
] = "Target"

# Assign identifier
data_dictionary.loc[
    data_dictionary["Feature"] == id_col,
    "Role"
] = "Identifier"

# Assign excluded features
data_dictionary.loc[
    data_dictionary["Feature"].isin(excluded_features),
    "Role"
] = "Excluded"

# Show remaining features requiring confirmation
print(data_dictionary["Role"].value_counts())

print("\nFeatures requiring review:")
print(
    data_dictionary.loc[
        data_dictionary["Role"] == "Review",
        "Feature"
    ].tolist()
)

In [ ]:
import joblib

# Load the shared preprocessing template
template_path = (
    project_root
    / "results"
    / "models"
    / "preprocessor_template.joblib"
)

print("Template exists:", template_path.exists())

preprocessor = joblib.load(template_path)

# Get columns used by the preprocessing template
model_features = []

for name, transformer, columns in preprocessor.transformers:
    if name == "remainder":
        continue

    if isinstance(columns, (list, tuple)):
        model_features.extend(columns)
    else:
        print("Please check column selector:", name, columns)

# Compare dataset features with template features
dataset_features = [
    col for col in df.columns
    if col not in ["row_id", "is_canceled"]
]

print("\nDataset candidate features:", len(dataset_features))
print("Template model features:", len(model_features))

print("\nFeatures in dataset but NOT in template:")
print(sorted(set(dataset_features) - set(model_features)))

print("\nFeatures in template but NOT in dataset:")
print(sorted(set(model_features) - set(dataset_features)))

print("\nDuplicate features in template:",
      len(model_features) != len(set(model_features)))

print("\nAll template features:")
print(model_features)

In [ ]:
# Define feature roles based on the saved preprocessing template

def assign_role(feature):
    if feature == "row_id":
        return "Identifier"
    elif feature == "is_canceled":
        return "Target"
    elif feature in model_features:
        return "Predictor"
    else:
        return "Excluded"

data_dictionary["Role"] = (
    data_dictionary["Feature"].apply(assign_role)
)

# Check role counts
print("Feature roles:")
print(data_dictionary["Role"].value_counts())

# Check excluded features
print("\nExcluded features:")
display(
    data_dictionary[
        data_dictionary["Role"] == "Excluded"
    ]
)

# Verify predictor consistency
dictionary_predictors = set(
    data_dictionary.loc[
        data_dictionary["Role"] == "Predictor",
        "Feature"
    ]
)

print(
    "\nDictionary matches template:",
    dictionary_predictors == set(model_features)
)

In [ ]:
# Feature descriptions for the cleaned dataset

feature_descriptions = {
    "row_id": "Unique row identifier created during data preparation.",
    "hotel": "Type of hotel: City Hotel or Resort Hotel.",
    "is_canceled": "Cancellation outcome: 1 = canceled, 0 = not canceled.",
    "lead_time": "Days between booking entry and scheduled arrival.",
    "arrival_date_year": "Scheduled arrival year.",
    "arrival_date_month": "Scheduled arrival month.",
    "arrival_date_week_number": "Week number of scheduled arrival.",
    "arrival_date_day_of_month": "Day of month of scheduled arrival.",
    "stays_in_weekend_nights": "Number of scheduled weekend nights.",
    "stays_in_week_nights": "Number of scheduled weekday nights.",
    "adults": "Number of adults in the booking.",
    "children": "Number of children in the booking.",
    "babies": "Number of babies in the booking.",
    "meal": "Meal plan associated with the booking.",
    "country": "Country of origin recorded for the guest.",
    "market_segment": "Market segment classification.",
    "distribution_channel": "Booking distribution channel.",
    "is_repeated_guest": "Whether the guest is recorded as a repeat guest.",
    "previous_cancellations": "Number of previous canceled bookings.",
    "previous_bookings_not_canceled": "Number of previous non-canceled bookings.",
    "reserved_room_type": "Room type originally reserved.",
    "deposit_type": "Deposit arrangement recorded for the booking.",
    "agent": "Travel agency identifier or missing-category value.",
    "customer_type": "Customer category recorded for the booking.",
    "adr": "Average daily rate recorded in the dataset.",
    "required_car_parking_spaces": "Number of requested parking spaces.",
    "total_of_special_requests": "Number of special requests recorded.",
    "total_guests": "Derived total number of adults, children, and babies.",
    "total_nights": "Derived total number of weekend and weekday nights.",
    "family_booking": "Derived indicator for bookings with children or babies.",
    "lead_time_group": "Derived category based on booking lead time.",
    "arrival_season": "Derived season based on scheduled arrival month."
}

data_dictionary["Description"] = (
    data_dictionary["Feature"]
    .map(feature_descriptions)
)

# Verify completeness
missing_descriptions = data_dictionary.loc[
    data_dictionary["Description"].isna(),
    "Feature"
].tolist()

print("Features without descriptions:", missing_descriptions)
print("Total documented features:", len(data_dictionary))

In [ ]:
# Document cleaning and modeling notes

cleaning_notes = {
    "row_id": "Keep for merging and split alignment; exclude from model input.",
    "is_canceled": "Prediction target; exclude from model input.",
    "children": "Missing values handled during data cleaning.",
    "country": "Missing values handled during data cleaning.",
    "agent": "Missing values represented as a separate category.",
    "total_guests": "Derived from adults, children, and babies.",
    "total_nights": "Derived from weekday and weekend nights.",
    "family_booking": "Derived from the presence of children or babies.",
    "lead_time_group": "Derived from lead_time.",
    "arrival_season": "Derived from arrival month; not included in current preprocessing template.",
    "required_car_parking_spaces": "Excluded from the primary classification template; retained in the raw and cleaned datasets."
}

# Fields whose availability at initial booking requires confirmation
review_features = {
    "adr",
    "deposit_type",
    "customer_type",
    "total_of_special_requests",
    "agent",
    "country",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "is_repeated_guest"
}

def build_cleaning_note(feature):
    notes = []

    if feature in cleaning_notes:
        notes.append(cleaning_notes[feature])

    if feature in review_features:
        notes.append(
            "Verify availability at initial booking before final modeling."
        )

    return " ".join(notes)

data_dictionary["Cleaning Notes"] = (
    data_dictionary["Feature"].apply(build_cleaning_note)
)

print("Dictionary shape:", data_dictionary.shape)

display(
    data_dictionary[
        [
            "Feature",
            "Role",
            "Description",
            "Cleaning Notes"
        ]
    ].head(10)
)

In [ ]:
# Final validation before exporting

expected_columns = set(df.columns)
dictionary_columns = set(data_dictionary["Feature"])

assert expected_columns == dictionary_columns, (
    "Dataset and dictionary columns do not match."
)

assert data_dictionary["Feature"].is_unique, (
    "Duplicate feature names found."
)

assert set(model_features) == set(
    data_dictionary.loc[
        data_dictionary["Role"] == "Predictor",
        "Feature"
    ]
), "Predictor roles do not match preprocessing template."

assert data_dictionary["Description"].notna().all(), (
    "Some feature descriptions are missing."
)

print("All data dictionary checks passed.")
print("Dataset features:", len(expected_columns))
print("Dictionary entries:", len(data_dictionary))
print("Model predictors:", len(model_features))

In [ ]:
# Define output path
dictionary_path = (
    project_root / "data" / "data_dictionary.csv"
)

# Export the data dictionary
data_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

# Verify exported file
dictionary_check = pd.read_csv(dictionary_path)

print("Saved to:", dictionary_path)
print("File exists:", dictionary_path.exists())
print("Exported shape:", dictionary_check.shape)

print(
    "All features preserved:",
    set(dictionary_check["Feature"]) == set(df.columns)
)

print(
    "All roles preserved:",
    dictionary_check["Role"].fillna("").tolist()
    == data_dictionary["Role"].fillna("").tolist()
)

In [ ]:
# Review features that may depend on booking-time availability

review_features = [
    "adr",
    "deposit_type",
    "customer_type",
    "total_of_special_requests",
    "agent",
    "country",
    "market_segment",
    "distribution_channel"
]

review_table = data_dictionary.loc[
    data_dictionary["Feature"].isin(review_features),
    ["Feature", "Role", "Description", "Cleaning Notes"]
].copy()

review_table["In Model Template"] = (
    review_table["Feature"].isin(model_features)
)

display(review_table)

In [ ]:
from pathlib import Path
import pandas as pd
import joblib

# Locate project files
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

template_path = (
    project_root / "results" / "models"
    / "preprocessor_template.joblib"
)

dictionary_path = (
    project_root / "data" / "data_dictionary.csv"
)

# Load the updated template and existing data dictionary
preprocessor = joblib.load(template_path)
data_dictionary = pd.read_csv(dictionary_path)

# Extract the official model input features
model_features = []

for name, transformer, columns in preprocessor.transformers:
    model_features.extend(list(columns))

# Update feature roles
data_dictionary["Role"] = "Excluded"

data_dictionary.loc[
    data_dictionary["Feature"] == "row_id", "Role"
] = "Identifier"

data_dictionary.loc[
    data_dictionary["Feature"] == "is_canceled", "Role"
] = "Target"

data_dictionary.loc[
    data_dictionary["Feature"].isin(model_features), "Role"
] = "Predictor"

# Verify consistency
assert len(model_features) == 24
assert data_dictionary["Role"].value_counts()["Predictor"] == 24
assert set(model_features).issubset(set(data_dictionary["Feature"]))

print("Updated feature roles:")
print(data_dictionary["Role"].value_counts())

print("\nExcluded features:")
print(
    data_dictionary.loc[
        data_dictionary["Role"] == "Excluded",
        "Feature"
    ].tolist()
)

# Save the updated dictionary
data_dictionary.to_csv(dictionary_path, index=False)

print("\nData dictionary updated successfully.")
print("Saved to:", dictionary_path)

In [ ]:
from pathlib import Path
import pandas as pd
import joblib

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

df = pd.read_csv(
    project_root / "data" / "hotel_bookings_clean.csv"
)

dictionary = pd.read_csv(
    project_root / "data" / "data_dictionary.csv"
)

template = joblib.load(
    project_root / "results" / "models"
    / "preprocessor_template.joblib"
)

model_features = []

for name, transformer, columns in template.transformers:
    model_features.extend(list(columns))

dictionary_features = dictionary.loc[
    dictionary["Role"] == "Predictor", "Feature"
].tolist()

# Check data, dictionary, and template consistency
assert set(df.columns) == set(dictionary["Feature"])
assert set(model_features) == set(dictionary_features)
assert len(model_features) == len(set(model_features)) == 24
assert df["row_id"].is_unique
assert df["is_canceled"].isin([0, 1]).all()
assert not df[model_features].isna().any().any()

print("Dataset rows:", len(df))
print("Dataset columns:", len(df.columns))
print("Model input features:", len(model_features))
print("Target values:", sorted(df["is_canceled"].unique()))
print("All consistency checks passed.")